In [ ]:
%pip install -q -e .[train]

In [ ]:
import mlflow
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset
from transformers import AutoTokenizer, TrainingArguments, Trainer

from utility import load
from utility.eval import macro_pr_auc
from utility.model import HFCrossEncoder, product_text
from utility.sampling import train_test_split

In [ ]:
load_dotenv()
mlflow.set_experiment('rubert-peft-and-aug')

env =   load()
repo_url = 'hf://datasets/' + env.config.data.data_repo
model_config = env.config.model.params
print(model_config)

In [ ]:
ATTR_CAP = 2000
TOTAL_CAP = 2500
SEED = model_config['seed']
TEST_SIZE = model_config['test_size']
LR = model_config['lr']
MAX_LEN = model_config['max_len']
TRAIN_BATCH = model_config['train_batch']
EVAL_BATCH = model_config['eval_batch']
TRAIN_EPOCHS = model_config['train_epochs']

LORA_R = model_config['model_lora_r']
LORA_ALPHA = model_config['model_lora_alpha']
LORA_DROPOUT = model_config['model_lora_dropout']
_start, _end = model_config['model_lora_layers'].split('-')
LORA_LAYERS = list(range(int(_start), int(_end)))
LORA_MODULES = model_config['model_lora_modules'].split(',')
MODEL_NAME = model_config['model_name']
SAVE_REPO = 'well-please/student_model'

SYMMETRY_RATIO = model_config['symmetry_prob']
LLM_SAMPLE_SIZE = model_config['llm_sample_size']
LABEL_SMOOTHING = model_config['label_smoothing']
LLM_FILL_WEIGHT = model_config['llm_fill_weight']

DEVICE = 'cuda' if torch.cuda.is_available() else ('xpu' if torch.xpu.is_available() else 'cpu')
print(f'Model: {MODEL_NAME}, LoRA r={LORA_R} alpha={LORA_ALPHA}, layers={LORA_LAYERS}, device={DEVICE}')
print(f'LLM sample: {LLM_SAMPLE_SIZE}, symmetry: {SYMMETRY_RATIO}, label_smoothing: {LABEL_SMOOTHING}')

In [ ]:
items_human = pl \
    .scan_parquet(f'{repo_url}/{env.config.data.items_human}') \
    .select(
        'id',
        pl.struct(['name', 'category', 'attributes'])
            .map_elements(lambda r: product_text(r['name'], r['category'], r['attributes'], attr_cap=ATTR_CAP, total_cap=TOTAL_CAP), return_dtype=pl.String).alias('text'),
        'category',
    )
matches = pl.scan_parquet(f'{repo_url}/{env.config.data.matches}')
human_pairs = matches \
    .join(items_human.select('id', pl.col('text').alias('text1'), pl.col('category').alias('category1')), right_on='id', left_on='id1') \
    .join(items_human.select('id', pl.col('text').alias('text2'), pl.col('category').alias('category2')), right_on='id', left_on='id2') \
    .unique()

print(f'human pairs: {human_pairs.select(pl.len()).collect().item()}')

stratification_columns = ('category1', 'category2', 'target')
train, test = train_test_split(human_pairs, stratification_columns, TEST_SIZE, SEED)
train, test = train.collect() if isinstance(train, pl.LazyFrame) else train, test.collect() if isinstance(test, pl.LazyFrame) else test

print(f'train: {train.shape}, test: {test.shape}')

In [ ]:
items_all = pl \
    .scan_parquet(f'{repo_url}/{env.config.data.items}') \
    .select(
        'id',
        pl.struct(['name', 'category', 'attributes'])
            .map_elements(lambda r: product_text(r['name'], r['category'], r['attributes'], attr_cap=ATTR_CAP, total_cap=TOTAL_CAP), return_dtype=pl.String).alias('text'),
        'category',
    )
matches_llm = pl.scan_parquet(f'{repo_url}/{env.config.data.matches_llm}')
llm_binary = matches_llm \
    .filter((pl.col('target') >= 0.75) | (pl.col('target') <= 0.25)) \
    .with_columns(pl.when(pl.col('target') >= 0.75).then(1.0).otherwise(0.0).alias('target')) \
    .head(LLM_SAMPLE_SIZE) \
    .join(items_all.select('id', pl.col('text').alias('text1'), pl.col('category').alias('category1')), right_on='id', left_on='id1') \
    .join(items_all.select('id', pl.col('text').alias('text2'), pl.col('category').alias('category2')), right_on='id', left_on='id2') \
    .unique() \
    .collect()

print(f'llm binary: {llm_binary.shape[0]}')
print(f'llm target distribution: {llm_binary["target"].mean():.3f}')

In [ ]:
CANONICAL_COLS = ['id1', 'id2', 'target', 'text1', 'category1', 'text2', 'category2', 'source', 'w']

train = train.with_columns(
    pl.lit('human').alias('source'), pl.lit(1.0).alias('w'),
)

forbidden = pl.concat([
    test.select(pl.col('id1').alias('id')),
    test.select(pl.col('id2').alias('id')),
]).unique()

llm_train = llm_binary \
    .join(forbidden, left_on='id1', right_on='id', how='anti') \
    .join(forbidden, left_on='id2', right_on='id', how='anti') \
    .with_columns(
        pl.col('target').cast(pl.Float64),
        pl.lit('llm').alias('source'),
        pl.lit(LLM_FILL_WEIGHT).alias('w'),
    )

train = pl.concat([train, llm_train]).select(CANONICAL_COLS)
train = train.sort(pl.col('text1').str.len_chars() + pl.col('text2').str.len_chars())

print(f'after concat: {train.shape}')
print(f'source distribution: {train["source"].value_counts()}')

idx = pl.int_range(0, pl.len()).shuffle(seed=SEED)
mask = idx < (pl.len() * SYMMETRY_RATIO)

symmetric = train.filter(mask).select(
    pl.col('id2').alias('id1'),
    pl.col('id1').alias('id2'),
    pl.col('target'),
    pl.col('text2').alias('text1'),
    pl.col('category2').alias('category1'),
    pl.col('text1').alias('text2'),
    pl.col('category1').alias('category2'),
    'source', 'w',
).select(CANONICAL_COLS)

train = pl.concat([train, symmetric]).sort(pl.col('text1').str.len_chars() + pl.col('text2').str.len_chars())

print(f'after symmetry: {train.shape} (+{symmetric.shape[0]} reversed pairs)')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = HFCrossEncoder.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_MODULES,
    layers_to_transform=LORA_LAYERS,
    lora_dropout=LORA_DROPOUT,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model = model.to(DEVICE)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, df: pl.DataFrame, tokenizer, max_len: int):
        self.text1 = df['text1'].to_list()
        self.text2 = df['text2'].to_list()
        self.targets = df['target'].to_list()
        self.categories = df['category1'].to_list()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        enc = self.tokenizer(
            self.text1[index],
            self.text2[index],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        enc['labels'] = torch.tensor(self.targets[index], dtype=torch.float)
        enc['category1'] = self.categories[index]
        return {k: v.squeeze(0) if isinstance(v, torch.Tensor) else v for k, v in enc.items()}


train_dataset = PairDataset(train, tokenizer, MAX_LEN)
test_dataset = PairDataset(test, tokenizer, MAX_LEN)
print(f'Datasets: train={len(train_dataset)}, test={len(test_dataset)}')

In [ ]:
class MetricTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        categories = inputs.pop('category1', None)
        labels = inputs.pop('labels')
        outputs = model(**inputs, labels=labels)
        logits = outputs['logits'] if isinstance(outputs, dict) else outputs

        loss = F.binary_cross_entropy_with_logits(logits, labels)
        if LABEL_SMOOTHING > 0:
            smooth = labels * (1 - LABEL_SMOOTHING) + 0.5 * LABEL_SMOOTHING
            loss = F.binary_cross_entropy_with_logits(logits, smooth)

        if self.state.global_step % 25 == 0 and categories is not None:
            with torch.no_grad():
                scores = logits.float().cpu().numpy()
                y_true = labels.float().cpu().numpy()
                cats = np.array(categories)
                self.log({'train_macro_pr_auc': macro_pr_auc(y_true, scores, cats)})

        return (loss, outputs) if return_outputs else loss


test_categories = np.array(test['category1'].to_list())


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    scores = logits.flatten()
    y_true = labels.flatten()
    return {'macro_pr_auc': macro_pr_auc(y_true, scores, test_categories)}


training_args = TrainingArguments(
    output_dir=env.data_dir,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=300,
    fp16=True,
    max_grad_norm=10.0,
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_pr_auc',
    greater_is_better=True,
    report_to='mlflow',
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = MetricTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
with mlflow.start_run(run_name='student-v8-lora', tags={'mlflow.user': 'jstnoname'}):
    mlflow.log_params({
        'model_name': MODEL_NAME,
        'seed': SEED,
        'lr': LR,
        'train_epochs': TRAIN_EPOCHS,
        'train_batch': TRAIN_BATCH,
        'eval_batch': EVAL_BATCH,
        'max_len': MAX_LEN,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'lora_layers': str(LORA_LAYERS),
        'lora_modules': str(LORA_MODULES),
        'symmetry_ratio': SYMMETRY_RATIO,
        'llm_sample_size': LLM_SAMPLE_SIZE,
        'label_smoothing': LABEL_SMOOTHING,
        'llm_fill_weight': LLM_FILL_WEIGHT,
        'train_shape': str(train.shape),
        'test_shape': str(test.shape),
    })
    trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
merged_model = model.merge_and_unload()

for attr in ('_hf_peft_config_loaded', 'peft_config'):
    if hasattr(merged_model, attr):
        delattr(merged_model, attr)

clean_state = {k.replace('base_model.model.', ''): v for k, v in merged_model.state_dict().items()}
merged_model.load_state_dict(clean_state)

unexpected = [k for k in clean_state if k.startswith('base_model')]
print(f'Prefix check: {len(unexpected)} bad keys')

merged_model.push_to_hub(SAVE_REPO, private=True, commit_message='student v8 - LoRA r64 + LLM 200K + symmetry + label smoothing')
tokenizer.push_to_hub(SAVE_REPO, private=True, commit_message='student v8 - LoRA r64 + LLM 200K + symmetry + label smoothing')
print(f'Model pushed to {SAVE_REPO}')